## Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:
* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.

The experiments can be explored by accessing the remote server.

The exampe uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file `mlflow_on_aws.md` to create a new AWS account and launch the tracking server. 

In [1]:
import mlflow
import os

#os.environ["AWS_PROFILE"] = "" # fill in with your AWS profile. More info: https://docs.aws.amazon.com/sdk-for-java/latest/developer-guide/setup.html#setup-credentials

TRACKING_SERVER_HOST = "172.191.91.229" # fill in with the public DNS of the EC2 instance
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://172.191.91.229:5000'


In [3]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/1', creation_time=1750225073825, experiment_id='1', last_update_time=1750225073825, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/0', creation_time=1750222976135, experiment_id='0', last_update_time=1750222976135, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="model")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2025/06/18 07:15:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 'wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/1/e80198e249da4e1f83fb6003f1940715/artifacts'
🏃 View run hilarious-cod-637 at: http://172.191.91.229:5000/#/experiments/1/runs/e80198e249da4e1f83fb6003f1940715
🧪 View experiment at: http://172.191.91.229:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/1', creation_time=1750225073825, experiment_id='1', last_update_time=1750225073825, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/0', creation_time=1750222976135, experiment_id='0', last_update_time=1750222976135, lifecycle_stage='active', name='Default', tags={}>]

In [6]:
print("Artifact URI:", mlflow.get_artifact_uri())


Artifact URI: wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/1/8144b75389ce4aa19742e5f0b51c3ccc/artifacts


### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [12]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1750231042323, description='', last_updated_timestamp=1750231043338, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1750231043338, current_stage='None', description='', last_updated_timestamp=1750231043338, name='iris-classifier', run_id='8144b75389ce4aa19742e5f0b51c3ccc', run_link='', source='wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/1/8144b75389ce4aa19742e5f0b51c3ccc/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='1'>], name='iris-classifier', tags={}>]

In [11]:
# Get the latest run from experiment '1'
runs = client.search_runs(experiment_ids=['1'], order_by=["start_time DESC"])
run_id = runs[0].info.run_id

mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2025/06/18 07:17:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1750231043338, current_stage='None', description='', last_updated_timestamp=1750231043338, name='iris-classifier', run_id='8144b75389ce4aa19742e5f0b51c3ccc', run_link='', source='wasbs://mlflow-artifacts@mlflowartifactsstore.blob.core.windows.net/1/8144b75389ce4aa19742e5f0b51c3ccc/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='1'>